# Notebook 04: Multi-Step Function Tools with the Responses API

This notebook shows how to run a small agent-style workflow with custom Python tools.

You will:
- define multiple function tools
- register Python implementations for those tools
- let the model call them through `create_function_tool_response(...)`
- return a final business-ready answer after the tool loop completes


## Why this notebook matters

Notebook 02 introduced tool calls conceptually. This notebook uses the shared helper in `utils.responses_api` so the model can plan, call tools, receive tool outputs, and continue to a final answer without manual loop code in the notebook.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

from utils import create_function_tool_response, extract_output_text
from utils.models import choose_default_model

PREFERENCE = 'reasoning'
MODEL_FOR_NOTEBOOK = choose_default_model(PREFERENCE)
print('Project root:', PROJECT_ROOT)
print('Using model:', MODEL_FOR_NOTEBOOK)


## Scenario

We will simulate a customer-success copilot. The model will need to gather account details and open support issues before it can recommend next steps and draft a follow-up email.

In [ ]:
ACCOUNT_DATA = {
    'Northwind Health': {
        'plan': 'Enterprise',
        'arr_usd': 185000,
        'renewal_date': '2026-05-30',
        'champion': 'Maya Chen',
        'health_score': 'yellow',
        'adoption_notes': 'Strong search usage but limited workflow automation adoption.',
    },
    'Blue Prairie Logistics': {
        'plan': 'Growth',
        'arr_usd': 48000,
        'renewal_date': '2026-07-15',
        'champion': 'Darius Cole',
        'health_score': 'green',
        'adoption_notes': 'Healthy weekly usage across operations and support.',
    },
}

TICKET_DATA = {
    'Northwind Health': [
        {
            'ticket_id': 'NW-401',
            'severity': 'high',
            'summary': 'SSO provisioning intermittently fails for new hires.',
            'status': 'open',
        },
        {
            'ticket_id': 'NW-418',
            'severity': 'medium',
            'summary': 'Finance team wants scheduled exports for audit reports.',
            'status': 'open',
        },
    ],
    'Blue Prairie Logistics': [
        {
            'ticket_id': 'BP-112',
            'severity': 'low',
            'summary': 'One user requested a dashboard color tweak.',
            'status': 'open',
        }
    ],
}


def lookup_account(account_name: str) -> dict:
    if account_name not in ACCOUNT_DATA:
        raise ValueError(f'Unknown account: {account_name}')
    return ACCOUNT_DATA[account_name]


def lookup_open_tickets(account_name: str) -> list[dict]:
    if account_name not in TICKET_DATA:
        return []
    return TICKET_DATA[account_name]


TOOLS = [
    {
        'type': 'function',
        'name': 'lookup_account',
        'description': 'Return CRM-style account details for a customer account.',
        'parameters': {
            'type': 'object',
            'properties': {
                'account_name': {'type': 'string'},
            },
            'required': ['account_name'],
            'additionalProperties': False,
        },
    },
    {
        'type': 'function',
        'name': 'lookup_open_tickets',
        'description': 'Return currently open support tickets for a customer account.',
        'parameters': {
            'type': 'object',
            'properties': {
                'account_name': {'type': 'string'},
            },
            'required': ['account_name'],
            'additionalProperties': False,
        },
    },
]

TOOL_FUNCTIONS = {
    'lookup_account': lookup_account,
    'lookup_open_tickets': lookup_open_tickets,
}


## Run the tool loop

The helper below does the standard loop for us:
1. send the initial request
2. detect `function_call` items
3. execute the mapped Python tools
4. continue the response until the model reaches a final answer


In [ ]:
prompt = """
You are a customer success strategist.

Review the account Northwind Health.
You must use both available tools before answering.

Return four sections:
1. Account snapshot
2. Renewal risk assessment
3. Recommended next 3 actions
4. Draft follow-up email to the account champion
""".strip()

final_response = create_function_tool_response(
    prompt,
    model=MODEL_FOR_NOTEBOOK,
    tools=TOOLS,
    tool_functions=TOOL_FUNCTIONS,
)

final_text = extract_output_text(final_response)
print(final_text)


## Optional inspection

The helper returns the final Responses API payload, so you can still inspect it if you want deeper visibility.

In [ ]:
print('Final response id:', getattr(final_response, 'id', 'n/a'))
print('Output item count:', len(getattr(final_response, 'output', []) or []))


## Deterministic tool sanity check

When building real agents, it helps to test the tools themselves separately from the model.

In [ ]:
print(json.dumps(lookup_account('Northwind Health'), indent=2))
print(json.dumps(lookup_open_tickets('Northwind Health'), indent=2))


## Takeaway

At this point, the notebook code is mostly business logic and schemas, not plumbing. That is the real value of moving the tool loop into shared utilities: the curriculum gets cleaner, and the same pattern becomes reusable in scripts and small apps.